# 07 — Graph-Level Metadata

0.3.3 introduced per-graph operational metadata materialized in the
registry graph:

- **Triple count** — how many triples does this graph hold?
- **Last modified** — when did the library most recently write to it?
- **Class inventory** — reified `cga:ClassInstanceCount` records
  mapping `(graph, rdf:type) → count`.

This notebook covers:

- Automatic metadata refresh in `eager` mode (the default)
- Explicit refresh in `off` mode
- Reading metadata via `get_graph_metadata()` and `get_holon_detail()`
- The rollup to per-holon `interiorTripleCount` + `holonLastModified`
- When metadata goes stale and how to reconcile


## Setup

Default mode is `eager` — metadata refreshes after every
library-mediated write.


In [ ]:
from holonic import HolonicDataset

ds = HolonicDataset()  # default: metadata_updates="eager"
ds.add_holon("urn:holon:sensor", "Sensor")
gi = ds.add_interior("urn:holon:sensor", '''
    @prefix ex: <urn:ex:> .
    <urn:track:1> a ex:Track ; ex:label "T1" .
    <urn:track:2> a ex:Track ; ex:label "T2" .
    <urn:track:3> a ex:Track ; ex:label "T3" .
    <urn:location:1> a ex:Location ; ex:label "L1" .
''')
print(f"interior graph: {gi}")


## Read metadata for a single graph


In [ ]:
md = ds.get_graph_metadata(gi)
print(f"triple count:      {md.triple_count}")
print(f"last modified:     {md.last_modified}")
print(f"class inventory:")
for c in md.class_inventory:
    print(f"  {c.class_iri}  -> {c.count}")


## Holon-level rollup

`get_holon_detail()` carries per-layer metadata and the rolled-up
`holon_last_modified` (max of layer lastModified values).


In [ ]:
detail = ds.get_holon_detail("urn:holon:sensor")
print(f"interior_triple_count: {detail.interior_triple_count}")
print(f"holon_last_modified:   {detail.holon_last_modified}")
print()
print("layer_metadata:")
for iri, md in detail.layer_metadata.items():
    print(f"  {iri}  triples={md.triple_count}  modified={md.last_modified}")


## Multi-interior holons

Metadata is tracked per graph AND rolled up at the holon level.


In [ ]:
ds.add_holon("urn:holon:multi", "Multi-source Holon")
ds.add_interior("urn:holon:multi", '''
    @prefix ex: <urn:ex:> .
    <urn:a:1> a ex:A . <urn:a:2> a ex:A .
''', graph_iri="urn:holon:multi/interior/source-a")
ds.add_interior("urn:holon:multi", '''
    @prefix ex: <urn:ex:> .
    <urn:b:1> a ex:B . <urn:b:2> a ex:B . <urn:b:3> a ex:B .
''', graph_iri="urn:holon:multi/interior/source-b")

detail = ds.get_holon_detail("urn:holon:multi")
print(f"total interior triples: {detail.interior_triple_count}")
for iri, md in detail.layer_metadata.items():
    print(f"  {iri}  {md.triple_count}")


## The "off" mode: callers refresh explicitly

For batch pipelines where eager refresh cost dominates, construct
with `metadata_updates="off"`. Nothing gets written until
`refresh_metadata(holon_iri)` or `refresh_all_metadata()` is called.


In [ ]:
batch_ds = HolonicDataset(metadata_updates="off")
batch_ds.add_holon("urn:holon:batch", "Batch")
gi = batch_ds.add_interior("urn:holon:batch", '''
    @prefix ex: <urn:ex:> .
    <urn:x:1> a ex:Item .
''')

# No metadata yet
print(f"before refresh: {batch_ds.get_graph_metadata(gi)}")

# Explicit refresh
batch_ds.refresh_metadata("urn:holon:batch")
md = batch_ds.get_graph_metadata(gi)
print(f"after refresh:  triples={md.triple_count}")


## Reconciling after out-of-band writes

Direct `backend.put_graph()` / `backend.update()` calls bypass the
eager refresh hook. Metadata goes stale. `refresh_metadata()` brings
it back to current.


In [ ]:
# Starts at 1 triple
md = ds.get_graph_metadata(gi) if ds.get_graph_metadata(gi) else None

# Out-of-band write — goes around the library's mutation API
batch_ds.backend.parse_into(
    "urn:holon:batch/interior",
    "@prefix ex: <urn:ex:> . <urn:x:2> a ex:Item . <urn:x:3> a ex:Item .",
    "turtle",
)
stale = batch_ds.get_graph_metadata("urn:holon:batch/interior")
print(f"stale read (metadata still shows old count): {stale.triple_count}")

# Reconcile
batch_ds.refresh_metadata("urn:holon:batch")
fresh = batch_ds.get_graph_metadata("urn:holon:batch/interior")
print(f"after refresh: {fresh.triple_count}")


## Refresh the whole holarchy

`refresh_all_metadata()` iterates every registered holon. Returns the
count refreshed.


In [ ]:
n = batch_ds.refresh_all_metadata()
print(f"refreshed {n} holon(s)")


## Where to go next

- `08_scope_resolution` — uses the class inventory for fast
  "has class X" discovery
- `10_projection_plugins` — projection runs honor `metadata_updates`
  when writing output graphs
